In [ ]:
import datetime
import os
from google.colab import drive
import numpy as np
import pandas as pd

In [ ]:
print("[STEP 1] Connecting to Google Drive...")
drive.mount('/content/drive')

[STEP 1] Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
FILE_PATH = (
    '/content/drive/MyDrive/smt 7/pakar/dataset_peredam_suara.xlsx'  # atau .csv jika csv
)

In [ ]:
if os.path.exists(FILE_PATH):
  print(f"File ditemukan: {FILE_PATH}")
  if FILE_PATH.endswith('.xlsx'):
    df_raw = pd.read_excel(FILE_PATH)
  else:
    df_raw = pd.read_csv(FILE_PATH)
  print(f"Total data berhasil di-load: {len(df_raw)} baris, {len(df_raw.columns)} kolom.")
else:
  print(f"File tidak ditemukan di path: {FILE_PATH}")
  print("Membuat dataframe darurat dari sampel...")

File ditemukan: /content/drive/MyDrive/smt 7/pakar/dataset_peredam_suara.xlsx
Total data berhasil di-load: 225 baris, 20 kolom.


In [ ]:
def fix_excel_date_corruption(val):
  """Fungsi untuk mengembalikan nilai desimal yang terkonversi otomatis menjadi tanggal oleh Excel (misal 2026-01-22 menjadi 22.1)."""
  if pd.isna(val):
    return np.nan

  # Jika tipe data adalah Timestamp/datetime (terkonversi oleh Excel)
  if isinstance(val, (pd.Timestamp, datetime.datetime)):
    day = val.day
    month = val.month
    # Excel konversi 22.1 -> 22 Jan -> day=22, month=1 -> 22.1
    return float(f'{day}.{month}')

  if isinstance(val, str):
    val_clean = val.replace(',', '.').strip()
    try:
      return float(val_clean)
    except ValueError:
      return val_clean

  return float(val)


print('\n[STEP 2] Cleaning Excel Auto-Date Corruption & Type Standardizations...')


[STEP 2] Cleaning Excel Auto-Date Corruption & Type Standardizations...


In [ ]:
df_clean = df_raw.copy()

# Daftar kolom numerik desimal yang berpotensi korup oleh Excel
NUMERICAL_FLOAT_COLS = [
    'luas_ruangan_m2',
    'ketebalan_dinding_cm',
    'luas_dinding_m2',
    'estimasi_noise_reduction_db',
    'estimasi_suara_di_luar_db',
    'jarak_hingga_40db_m',
]

# Terapkan pembersihan pada kolom-kolom terkait
for col in NUMERICAL_FLOAT_COLS:
  if col in df_clean.columns:
    df_clean[col] = df_clean[col].apply(fix_excel_date_corruption)
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

# Pastikan kolom numerik bulat berbentuk integer
NUMERICAL_INT_COLS = [
    'id',
    'intensitas_suara_awal_db',
    'target_suara_db',
    'ketebalan_peredam_mm',
    'estimasi_biaya_rp',
]
for col in NUMERICAL_INT_COLS:
  if col in df_clean.columns:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0).astype(int)

# PHASE 1: REKONSTRUKSI FISIKA AKUSTIK (ASTM E336)

In [ ]:

import numpy as np
import pandas as pd

print("[STEP 3 REVISED] Reconstructing Physics Features via ASTM E336...")

# 1. Selisih Reduksi Kebisingan yang dibutuhkan (Target NR)
df_clean['delta_db'] = (
    df_clean['intensitas_suara_awal_db'] - df_clean['target_suara_db']
)

# 2. Estimasi Absorpsi Ruang Penerima (A) berdasarkan Sabine Approximation: A ≈ 0.20 * Volume/Luas
# Diasumsikan koefisien serap rata-rata ruangan α ≈ 0.20
df_clean['estimasi_absorpsi_A'] = df_clean['luas_ruangan_m2'] * 0.20 + 5.0

# 3. Hitung Koreksi Geometri ASTM E336: 10 * log10(S / A)
# S = Luas Dinding Pembatas (luas_dinding_m2)
df_clean['koreksi_geometri_db'] = 10 * np.log10(
    df_clean['luas_dinding_m2'] / df_clean['estimasi_absorpsi_A']
)

# 4. Hitung Daya Redam Murni Material yang Dibutuhkan (TL Required)
# Formula ASTM E336: TL = NR + 10 * log10(S / A)
df_clean['tl_required_db'] = (
    df_clean['delta_db'] + df_clean['koreksi_geometri_db']
)

# 5. Verifikasi Suara di Luar/Ruang Penerima berbasis NR Aktual
df_clean['estimasi_suara_di_luar_db'] = (
    df_clean['intensitas_suara_awal_db']
    - df_clean['estimasi_noise_reduction_db']
).clip(lower=0)

print("Kalkulasi fisika ASTM E336 berhasil ditambahkan.")

[STEP 3 REVISED] Reconstructing Physics Features via ASTM E336...
Kalkulasi fisika ASTM E336 berhasil ditambahkan.


In [ ]:
print('\n[STEP 4] Defining Problem & Solution Vectors for ICBR...')

PROBLEM_ATTRIBUTES = [
    'luas_ruangan_m2',
    'jenis_dinding',
    'ketebalan_dinding_cm',
    'material_dinding',
    'luas_dinding_m2',
    'sumber_suara',
    'intensitas_suara_awal_db',
    'target_suara_db',
    'delta_db',
    'frekuensi_dominan',
]

SOLUTION_ATTRIBUTES = [
    'material_peredam',
    'ketebalan_peredam_mm',
    'sistem_pemasangan',
    'estimasi_noise_reduction_db',
    'estimasi_suara_di_luar_db',
    'jarak_hingga_40db_m',
    'estimasi_biaya_rp',
    'tingkat_rekomendasi',
]

print(
    f"Problem Vector (P) contains {len(PROBLEM_ATTRIBUTES)} attributes:"
    f" {PROBLEM_ATTRIBUTES}"
)
print(
    f"Solution Vector (S) contains {len(SOLUTION_ATTRIBUTES)} attributes:"
    f" {SOLUTION_ATTRIBUTES}"
)


[STEP 4] Defining Problem & Solution Vectors for ICBR...
Problem Vector (P) contains 10 attributes: ['luas_ruangan_m2', 'jenis_dinding', 'ketebalan_dinding_cm', 'material_dinding', 'luas_dinding_m2', 'sumber_suara', 'intensitas_suara_awal_db', 'target_suara_db', 'delta_db', 'frekuensi_dominan']
Solution Vector (S) contains 8 attributes: ['material_peredam', 'ketebalan_peredam_mm', 'sistem_pemasangan', 'estimasi_noise_reduction_db', 'estimasi_suara_di_luar_db', 'jarak_hingga_40db_m', 'estimasi_biaya_rp', 'tingkat_rekomendasi']


In [ ]:
print('\n[STEP 5] Final Quality Check & Export Clean Data...')

# Cek missing values
missing_count = df_clean.isnull().sum().sum()
print(f"Total Missing Values: {missing_count}")

# Tampilkan 5 baris pertama data bersih
print('\nSample 5 Baris Data Bersih:')
print(
    df_clean[
        [
            'id',
            'luas_ruangan_m2',
            'delta_db',
            'material_peredam',
            'estimasi_noise_reduction_db',
            'estimasi_suara_di_luar_db',
            'jarak_hingga_40db_m',
        ]
    ].head()
)

# Simpan ke Google Drive sebagai CSV bersih
OUTPUT_PATH = '/content/drive/MyDrive/smt 7/pakar/peredam_suara_clean.csv'
df_clean.to_csv(OUTPUT_PATH, index=False)
print(
    f"\nSUCCESS! Data bersih berhasil disimpan di Drive: {OUTPUT_PATH}"
)


[STEP 5] Final Quality Check & Export Clean Data...
Total Missing Values: 0

Sample 5 Baris Data Bersih:
   id  luas_ruangan_m2  delta_db       material_peredam  \
0   1             52.8        23     Glasswool 24 kg/m3   
1   2             16.8        28     Glasswool 24 kg/m3   
2   3            117.4        49      Rockwool 60 kg/m3   
3   4             20.4        32  Greenwool / PET fiber   
4   5             74.0        55      Rockwool 60 kg/m3   

   estimasi_noise_reduction_db  estimasi_suara_di_luar_db  jarak_hingga_40db_m  
0                         40.9                       22.1                  0.0  
1                         43.3                       14.7                  0.0  
2                         46.6                       47.4                  2.3  
3                         33.5                       28.5                  0.0  
4                         38.6                       61.4                 11.7  

SUCCESS! Data bersih berhasil disimpan di Drive: /co

In [ ]:
from scipy.optimize import minimize

In [ ]:
CSV_PATH = '/content/drive/MyDrive/smt 7/pakar/peredam_suara_clean.csv'

In [ ]:
df_cases = pd.read_csv(CSV_PATH)
print(
    f"Dataset berhasil dimuat: {len(df_cases)} kasus, {len(df_cases.columns)}"
    " kolom.\n"
)

Dataset berhasil dimuat: 225 kasus, 24 kolom.



In [ ]:
FEATURE_COLS = [
    'luas_ruangan_m2',
    'ketebalan_dinding_cm',
    'luas_dinding_m2',
    'intensitas_suara_awal_db',
    'target_suara_db',
    'delta_db',
]

In [ ]:
def extract_normalized_matrix(df, feature_cols):
  """Mengekstrak dan meng-skala matriks fitur V ke rentang [0, 1] (Min-Max Scaling)."""
  V_raw = df[feature_cols].values.astype(float)
  v_min = V_raw.min(axis=0)
  v_max = V_raw.max(axis=0)

  # Mencegah pembagian dengan nol jika min == max
  v_denom = np.where((v_max - v_min) == 0, 1.0, (v_max - v_min))
  V_norm = (V_raw - v_min) / v_denom

  return V_raw, V_norm, v_min, v_max

In [ ]:
V_raw, V_norm, v_min, v_max = extract_normalized_matrix(df_cases, FEATURE_COLS)
k_cases, m_features = V_norm.shape
print(
    f"Matriks Fitur Problem (V) terbentuk: {k_cases} kasus x {m_features}"
    " atribut numerik."
)

Matriks Fitur Problem (V) terbentuk: 225 kasus x 6 atribut numerik.


# PHASE 2: CORE RETRIEVAL ENGINE (CRNHW - ASTM E336)

## STEP 2: AHP SUBJECTIVE WEIGHTS (Ws) - MATRIKS TERKOREKSI ASTM E336

In [ ]:
J_expert = np.array([
    # [0]   [1]   [2]   [3]   [4]   [5]
    [1.00, 2.00, 0.33, 0.33, 0.50, 0.25],  # [0] luas_ruangan_m2
    [0.50, 1.00, 0.25, 0.25, 0.33, 0.20],  # [1] ketebalan_dinding_cm
    [3.00, 4.00, 1.00, 0.50, 2.00, 0.33],  # [2] luas_dinding_m2 (S - Koreksi ASTM E336)
    [3.00, 4.00, 2.00, 1.00, 2.00, 0.50],  # [3] intensitas_suara_awal_db
    [2.00, 3.00, 0.50, 0.50, 1.00, 0.25],  # [4] target_suara_db
    [4.00, 5.00, 3.00, 2.00, 4.00, 1.00],  # [5] delta_db (Target Utama NR_req)
])

In [ ]:
def calculate_ahp_weights(J_matrix):
  """Menghitung Bobot Subjektif (Ws) dan Consistency Ratio (CR) menggunakan AHP."""
  m = J_matrix.shape[0]

  # Geometric Mean Method
  geom_mean = np.exp(np.mean(np.log(J_matrix), axis=1))
  Ws = geom_mean / np.sum(geom_mean)

  # Check Consistency Ratio (CR)
  lambda_max = np.mean((J_matrix @ Ws) / Ws)
  CI = (lambda_max - m) / (m - 1) if m > 1 else 0.0

  # Random Index (RI) Table untuk order 1 s/d 10
  RI_table = {
      1: 0.0,
      2: 0.0,
      3: 0.58,
      4: 0.89,
      5: 1.12,
      6: 1.26,
      7: 1.36,
      8: 1.41,
      9: 1.46,
      10: 1.49,
  }
  RI = RI_table.get(m, 1.26)
  CR = CI / RI if RI > 0 else 0.0

  return Ws, CR
Ws, CR = calculate_ahp_weights(J_expert)
print(f"\n[STEP 2] AHP Subjective Weights (Ws) Calculated (CR = {CR:.4f}):")
for col, w in zip(FEATURE_COLS, Ws):
  print(f"  - {col:25s}: {w:.4f} ({w*100:.2f}%)")


[STEP 2] AHP Subjective Weights (Ws) Calculated (CR = 0.0252):
  - luas_ruangan_m2          : 0.0730 (7.30%)
  - ketebalan_dinding_cm     : 0.0475 (4.75%)
  - luas_dinding_m2          : 0.1675 (16.75%)
  - intensitas_suara_awal_db : 0.2262 (22.62%)
  - target_suara_db          : 0.1131 (11.31%)
  - delta_db                 : 0.3727 (37.27%)


## STEP 3: ENTROPY OBJECTIVE WEIGHTS (Wo)

In [ ]:
def calculate_entropy_weights(V_matrix):
  k, m = V_matrix.shape
  eps = 1e-12
  column_sums = np.sum(V_matrix, axis=0) + eps
  Q = V_matrix / column_sums
  e_o = -1.0 / np.log(k) * np.sum(Q * np.log(Q + eps), axis=0)
  g_j = 1.0 - e_o
  Wo = g_j / np.sum(g_j)
  return Wo

In [ ]:
Wo = calculate_entropy_weights(V_norm)
print("\n[STEP 3] Entropy Objective Weights (Wo) Calculated:")
for col, w in zip(FEATURE_COLS, Wo):
  print(f"  - {col:25s}: {w:.4f} ({w*100:.2f}%)")


[STEP 3] Entropy Objective Weights (Wo) Calculated:
  - luas_ruangan_m2          : 0.2219 (22.19%)
  - ketebalan_dinding_cm     : 0.2069 (20.69%)
  - luas_dinding_m2          : 0.1491 (14.91%)
  - intensitas_suara_awal_db : 0.1368 (13.68%)
  - target_suara_db          : 0.1553 (15.53%)
  - delta_db                 : 0.1299 (12.99%)


## STEP 4: HYBRID WEIGHT OPTIMIZATION (Wh)

In [ ]:
def calculate_hybrid_weights(Ws, Wo, V_matrix):
  k, m = V_matrix.shape

  def objective_func(Wh):
    dev_s = np.sum(((Wh - Ws) * V_matrix) ** 2)
    dev_o = np.sum(((Wh - Wo) * V_matrix) ** 2)
    return dev_s + dev_o

  constraints = {'type': 'eq', 'fun': lambda Wh: np.sum(Wh) - 1.0}
  bounds = [(0.0, 1.0) for _ in range(m)]
  Wh_init = (Ws + Wo) / 2.0

  opt_result = minimize(
      objective_func, Wh_init, method='SLSQP', bounds=bounds, constraints=constraints
  )
  return opt_result.x

In [ ]:
Wh = calculate_hybrid_weights(Ws, Wo, V_norm)

print('\n[STEP 4] Optimized Hybrid Weights (Wh) Result:')
for col, w in zip(FEATURE_COLS, Wh):
  print(f"  - {col:25s}: {w:.4f} ({w*100:.2f}%)")


[STEP 4] Optimized Hybrid Weights (Wh) Result:
  - luas_ruangan_m2          : 0.1475 (14.75%)
  - ketebalan_dinding_cm     : 0.1272 (12.72%)
  - luas_dinding_m2          : 0.1583 (15.83%)
  - intensitas_suara_awal_db : 0.1815 (18.15%)
  - target_suara_db          : 0.1342 (13.42%)
  - delta_db                 : 0.2513 (25.13%)


## STEP 5: WEIGHTED KNN RETRIEVAL ENGINE

In [ ]:
def retrieve_top_k_cases(new_problem_dict, df_base, Wh, K=10):
  p0_raw = np.array([new_problem_dict[col] for col in FEATURE_COLS])
  v_denom = np.where((v_max - v_min) == 0, 1.0, (v_max - v_min))
  p0_norm = (p0_raw - v_min) / v_denom

  diff_matrix = np.abs(V_norm - p0_norm)
  sim_num_matrix = 1.0 - diff_matrix

  sim_cat_dinding = (
      df_base['jenis_dinding'].str.lower().values
      == str(new_problem_dict['jenis_dinding']).lower()
  ).astype(float)
  sim_cat_sumber = (
      df_base['sumber_suara'].str.lower().values
      == str(new_problem_dict['sumber_suara']).lower()
  ).astype(float)

  weighted_sim_num = np.dot(sim_num_matrix, Wh) / np.sum(Wh)
  total_similarity = 0.80 * weighted_sim_num + 0.10 * sim_cat_dinding + 0.10 * sim_cat_sumber

  df_results = df_base.copy()
  df_results['similarity_score'] = total_similarity
  df_results['similarity_pct'] = np.round(total_similarity * 100, 2)

  return df_results.sort_values(by='similarity_score', ascending=False).head(K)

In [ ]:
# Testing dengan Sampel User Baru
sample_user_input = {
    'luas_ruangan_m2': 40.0,
    'jenis_dinding': 'Hebel',
    'ketebalan_dinding_cm': 10.0,
    'material_dinding': 'Bata ringan AAC',
    'luas_dinding_m2': 60.0,
    'sumber_suara': 'Musik / audio',
    'intensitas_suara_awal_db': 85,
    'target_suara_db': 40,
    'delta_db': 85 - 40,  # 45 dB
    'frekuensi_dominan': '250 Hz-4 kHz',
}

top_10_candidates = retrieve_top_k_cases(sample_user_input, df_cases, Wh, K=10)
print("\n[SUCCESS] Top 10 Candidates Retrieved for Phase 3!")


[SUCCESS] Top 10 Candidates Retrieved for Phase 3!


# PHASE 3: CASE ADAPTATION ENGINE (CAHWM - ASTM E336 REVISED)

## STEP 1: VALIDASI INPUT DARI RETRIEVAL ENGINE (PHASE 2)

In [ ]:
print("[STEP 1] Validasi Input Top-K Candidate Cases...")
if 'top_10_candidates' not in locals():
  raise NameError(
      "Variabel 'top_10_candidates' tidak ditemukan. Jalankan kode Phase 2"
      " terlebih dahulu!"
  )

K_candidates = len(top_10_candidates)
print(f"Menggunakan {K_candidates} kasus kandidat teratas untuk adaptasi.\n")

[STEP 1] Validasi Input Top-K Candidate Cases...
Menggunakan 10 kasus kandidat teratas untuk adaptasi.



## STEP 2: DEFINISI STRUKTUR MATRIKS CAHWM

In [ ]:
PROBLEM_NUM_COLS = [
    'luas_ruangan_m2',
    'ketebalan_dinding_cm',
    'luas_dinding_m2',
    'intensitas_suara_awal_db',
    'target_suara_db',
    'delta_db',
]

# Atribut Solusi Numerik (n = 4)
SOLUTION_NUM_COLS = [
    'ketebalan_peredam_mm',
    'estimasi_noise_reduction_db',
    'estimasi_suara_di_luar_db',
    'estimasi_biaya_rp',
]

m_prob = len(PROBLEM_NUM_COLS)
n_sol = len(SOLUTION_NUM_COLS)

## STEP 3: BENTUK MATRIKS KEMIRIPAN (S) & VEKTOR UTILITAS (U)

In [ ]:
print("[STEP 2] Membentuk Matriks Kemiripan (S) & Vektor Utilitas (U)...")

# 1. Matriks Kemiripan S (K x m) - Normalisasi Min-Max
S_raw = top_10_candidates[PROBLEM_NUM_COLS].values.astype(float)
s_min = S_raw.min(axis=0)
s_max = S_raw.max(axis=0)
s_denom = np.where((s_max - s_min) == 0, 1.0, (s_max - s_min))
S = (S_raw - s_min) / s_denom

# 2. Vektor Utilitas U (K x 1)
u_k = np.dot(S, Wh)
eps = 1e-12
u_norm = u_k / (np.sum(u_k) + eps)  # \hat{u}_k (Normalized Utility)

print(f"  - Matriks Kemiripan S terkonstruksi (Ukuran: {S.shape})")
print(f"  - Vektor Utilitas U terhitung (Total: {np.sum(u_norm):.2f})\n")

[STEP 2] Membentuk Matriks Kemiripan (S) & Vektor Utilitas (U)...
  - Matriks Kemiripan S terkonstruksi (Ukuran: (10, 6))
  - Vektor Utilitas U terhitung (Total: 1.00)



## STEP 4: GREY RELATIONAL ANALYSIS MATRIX (R)

In [ ]:
print("[STEP 3] Menghitung Matriks Relasi Grey (R)...")

# Normalisasi relatif terhadap kandidat pertama
p_2 = S_raw / (S_raw[0, :] + eps)
Sol_raw = top_10_candidates[SOLUTION_NUM_COLS].values.astype(float)
s_2 = Sol_raw / (Sol_raw[0, :] + eps)

xi = 0.5  # Distinguishing coefficient
R = np.zeros((m_prob, n_sol))

for i in range(m_prob):
  for j in range(n_sol):
    diff = np.abs(p_2[:, i] - s_2[:, j])
    min_diff = np.min(diff)
    max_diff = np.max(diff)

    grey_coef = (min_diff + xi * max_diff) / (diff + xi * max_diff + eps)
    # R_ij = sum_z (\hat{u}_z * r_zij)
    R[i, j] = np.sum(u_norm * grey_coef)

print(f"  - Matriks Relasi Grey R terhitung (Ukuran {R.shape})\n")


[STEP 3] Menghitung Matriks Relasi Grey (R)...
  - Matriks Relasi Grey R terhitung (Ukuran (6, 4))



## STEP 5: MATRIKS ADAPTABILITAS (A) & CONSTRAINT CHECK

In [ ]:
print("[STEP 4] Menganalisis Adaptabilitas & Kendala Teknis (A)...")

sim_scores = top_10_candidates['similarity_score'].values
A = np.zeros((K_candidates, n_sol))

for k_idx in range(K_candidates):
  row_cand = top_10_candidates.iloc[k_idx]
  constraint_penalty = 1.0

  # Penalti 1: Jika tebal peredam > 75mm padahal luas ruangan kecil (< 25m2)
  if (
      row_cand['ketebalan_peredam_mm'] > 75
      and sample_user_input['luas_ruangan_m2'] < 25.0
  ):
    constraint_penalty *= 0.85

  # Penalti 2: Jika Noise Reduction kandidat kurang dari target ΔdB user
  if row_cand['estimasi_noise_reduction_db'] < sample_user_input['delta_db']:
    constraint_penalty *= 0.90

  for j in range(n_sol):
    A[k_idx, j] = 0.5 * sim_scores[k_idx] + 0.5 * constraint_penalty

print(f"  - Matriks Adaptabilitas A terkonstruksi (Ukuran {A.shape})\n")

[STEP 4] Menganalisis Adaptabilitas & Kendala Teknis (A)...
  - Matriks Adaptabilitas A terkonstruksi (Ukuran (10, 4))



## STEP 6: HYBRID WEIGHTED MEAN (HWM) MATRIX & ADAPTED SOLUTION

In [ ]:
print("[STEP 5] Menghitung Hybrid Weighted Mean (HWM)...")

# WM = (S x R) Hadamard A
SR_matrix = np.dot(S, R)  # (K x n)
WM = SR_matrix * A  # Hadamard Product (K x n)

# Normalisasi Matriks WM -> HWM
HWM = WM / (np.sum(WM, axis=0) + eps)

# 1. Ekstraksi Solusi Numerik (Kontinu)
adapted_numerical_solution = np.sum(HWM * Sol_raw, axis=0)
adapted_sol_dict = {
    col: adapted_numerical_solution[idx]
    for idx, col in enumerate(SOLUTION_NUM_COLS)
}


# 2. Ekstraksi Solusi Kategorikal via Weighted Voting
def adapt_categorical_attribute(top_df, col_name):
  weighted_votes = {}
  for _, row in top_df.iterrows():
    val = row[col_name]
    score = row['similarity_score']
    weighted_votes[val] = weighted_votes.get(val, 0.0) + score
  return max(weighted_votes, key=weighted_votes.get)


adapted_material = adapt_categorical_attribute(
    top_10_candidates, 'material_peredam'
)
adapted_sistem = adapt_categorical_attribute(
    top_10_candidates, 'sistem_pemasangan'
)

# Klasifikasi Tingkat Rekomendasi
max_sim = top_10_candidates['similarity_score'].max()
if max_sim >= 0.85:
  tingkat_rekomendasi = 'Tinggi'
elif max_sim >= 0.70:
  tingkat_rekomendasi = 'Sedang'
else:
  tingkat_rekomendasi = 'Rendah'

[STEP 5] Menghitung Hybrid Weighted Mean (HWM)...


## STEP 7: PHYSICAL BOUNDARY CONSTRAINTS & ASTM E336 FIELD CALCULATION

In [ ]:
print("[STEP 6] Applying Physical Boundary Constraints & ASTM E336 Rules...")

# --- ATURAN FISIKA 1: FLANKING TRANSMISSION OVERRIDE ---
# Pemasangan "Ditempel langsung" mengalami kebocoran getaran struktural.
# Jika target peredaman ΔdB > 25 dB, WAJIB menggunakan Air Gap / Rangka Ganda.
if sample_user_input['delta_db'] > 25.0:
  if adapted_sistem == 'Ditempel langsung ke dinding':
    adapted_sistem = 'Peredam + rongga udara (air gap 5 cm) + gypsum 12mm'

# --- ATURAN FISIKA 2: ASTM E336 NOISE REDUCTION (NR = TL - 10 log10(S/A)) ---
# S = Luas Dinding Pembatas (m2)
# A = Estimasi Absorpsi Ruang Penerima (m2 Sabin) -> Sabine Approx: A ≈ 0.20 * Volume + 5
S_user = sample_user_input['luas_dinding_m2']
A_user = sample_user_input['luas_ruangan_m2'] * 0.20 + 5.0
koreksi_geometri_user = 10 * np.log10(S_user / A_user)

# TL Material hasil adaptasi HWM (murni sifat material)
tl_material_hwm = round(adapted_sol_dict['estimasi_noise_reduction_db'], 1)

# Noise Reduction (NR) Aktual Lapangan
est_nr_aktual = round(tl_material_hwm - koreksi_geometri_user, 1)

# Estimasi Suara di Ruang Penerima (Suara Luar)
est_suara_luar = max(
    0.0, round(sample_user_input['intensitas_suara_awal_db'] - est_nr_aktual, 1)
)

# Ketebalan & Biaya
tebal_mm = round(adapted_sol_dict['ketebalan_peredam_mm'] / 5) * 5
est_biaya_total = int(
    np.round(adapted_sol_dict['estimasi_biaya_rp'] / 50000) * 50000
)
biaya_per_m2 = int(round(est_biaya_total / sample_user_input['luas_dinding_m2']))



[STEP 6] Applying Physical Boundary Constraints & ASTM E336 Rules...

[HASIL ADAPTASI SOLUSI AKHIR (OUTPUT SISTEM - ASTM E336)]
• Material Peredam          : Rockwool 60 kg/m3
• Ketebalan Peredam         : 60 mm
• Sistem Pemasangan         : Peredam + rongga udara (air gap 5 cm) + gypsum 12mm (Disesuaikan via Flanking Rule)
• Estimasi TL Material (HWM): 43.2 dB
• Koreksi Geometri (10log S/A): 6.64 dB
• Estimasi NR Lapangan      : 36.6 dB
• Estimasi Suara di Luar    : 48.4 dB
• Estimasi Total Biaya      : Rp 19,400,000
• Estimasi Biaya per m²     : Rp 323,333 / m²
• Tingkat Rekomendasi       : Tinggi (Skor Kemiripan Max: 86.31%)


## OUTPUT REKOMENDASI SISTEM

In [ ]:

print('\n' + '=' * 70)
print('[HASIL ADAPTASI SOLUSI AKHIR (OUTPUT SISTEM - ASTM E336)]')
print('=' * 70)
print(f'• Material Peredam          : {adapted_material}')
print(f'• Ketebalan Peredam         : {tebal_mm} mm')
print(
    f'• Sistem Pemasangan         : {adapted_sistem} (Disesuaikan via Flanking'
    ' Rule)'
)
print(f'• Estimasi TL Material (HWM): {tl_material_hwm} dB')
print(f'• Koreksi Geometri (10log S/A): {koreksi_geometri_user:.2f} dB')
print(f'• Estimasi NR Lapangan      : {est_nr_aktual} dB')
print(f'• Estimasi Suara di Luar    : {est_suara_luar} dB')
print(f'• Estimasi Total Biaya      : Rp {est_biaya_total:,}')
print(f'• Estimasi Biaya per m²     : Rp {biaya_per_m2:,} / m²')
print(
    f'• Tingkat Rekomendasi       : {tingkat_rekomendasi} (Skor Kemiripan'
    f' Max: {max_sim*100:.2f}%)'
)
print('=' * 70)